In [1]:
!pip install --upgrade pip setuptools wheel
!pip install seqeval evaluate tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 51.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 40.6 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=0599ade060643bfc13ac020ebdb26c7266a2b78675bb45b7649c2681eb6ab0ae
  Stored in directory: /root/.cache/pip/w

In [3]:
import os
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


In [4]:
# Reproducibility
SEED = 23022006
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA devices:", torch.cuda.device_count())

Device: cuda
GPU: Tesla T4
CUDA devices: 2


In [5]:
# Config
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

MAX_LENGTH = 258
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
EPOCHS = 10
VALID_RATIO = 0.1

LR_BERT = 2e-5
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-3
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
HIDDEN_DIM = 32
DROPOUT_RATE = 0.5

OUTPUT_DIR = Path("/kaggle/working") 
BEST_MODEL_PATH = OUTPUT_DIR / "biomedbert_bilstm_crf_ddi_best.pt"
ENTITY_TYPES = ['drug', 'brand', 'group', 'drug_n']


In [6]:
TRAIN_DIR_DrugBank = Path("/kaggle/input/datasets/tuantc2306/trainingset/DDICorpus/Train/DrugBank")
TRAIN_DIR_MedLine = Path("/kaggle/input/datasets/tuantc2306/trainingset/DDICorpus/Train/MedLine")

TEST_ROOT = Path("/kaggle/input/datasets/tuantc2306/trainingset/DDICorpus/Test")
TEST_DIR = TEST_ROOT / "Test for DrugNER task"

In [7]:
from pathlib import Path
from sklearn.model_selection import train_test_split


# 1. Chia danh sách file của từng nguồn
def split_xml_files(folder, dev_size=0.1, seed=42):
    xml_files = sorted(Path(folder).rglob("*.xml"))

    train_files, dev_files = train_test_split(
        xml_files,
        test_size=dev_size,
        random_state=seed,
        shuffle=True,
    )

    return train_files, dev_files
train_files_drug, dev_files_drug = split_xml_files(
    TRAIN_DIR_DrugBank, seed= SEED
)

train_files_med, dev_files_med = split_xml_files(
    TRAIN_DIR_MedLine, seed= SEED
)
test_files = sorted(TEST_DIR.rglob("*.xml"))

In [8]:
def read_xml_file(path):
    examples = []
    for sent in ET.parse(path).getroot().iter("sentence"):
        text = sent.attrib.get("text", "")
        if not text.strip():
            if sent.findall("entity"):
                raise ValueError(f"Annotation trong câu rỗng: {path}")
            continue
        entities = []
        for ent in sent.findall("entity"):
            kind = ent.attrib["type"]
            if kind not in ENTITY_TYPES:
                raise ValueError(f"Loại entity không hỗ trợ: {kind}")
            for span in ent.attrib["charOffset"].split(";"):
                start, last = map(int, span.split("-"))
                end = last + 1
                if not 0 <= start < end <= len(text):
                    raise ValueError(f"Offset XML không hợp lệ: {path}, {span}")
                entities.append(dict(start=start, end=end, type=kind, text=text[start:end]))
        examples.append(dict(id=sent.attrib.get("id", ""), document=str(path),
                             text=text, entities=entities))
    return examples

# 2. Đọc các câu từ danh sách file đã chia
def load_examples_from_files(xml_files):
    examples = []

    for xml_file in xml_files:
        examples.extend(read_xml_file(xml_file))

    return examples
train_examples_drug = load_examples_from_files(train_files_drug)
dev_examples_drug = load_examples_from_files(dev_files_drug)

train_examples_med = load_examples_from_files(train_files_med)
dev_examples_med = load_examples_from_files(dev_files_med)
# 3. Gộp hai nguồn
train_examples = train_examples_drug + train_examples_med
dev_examples = dev_examples_drug + dev_examples_med

# 4. Giữ nguyên tập test có sẵn
test_examples = load_examples_from_files(test_files)

In [9]:
# 5. Kiểm tra không trùng file giữa train và dev
train_files = train_files_drug + train_files_med
dev_files = dev_files_drug + dev_files_med

assert set(train_files).isdisjoint(dev_files)

for source, train_part, dev_part in [
    ("DrugBank", train_examples_drug, dev_examples_drug),
    ("MedLine", train_examples_med, dev_examples_med),
]:
    total = len(train_part) + len(dev_part)
    print(
        f"{source}: train={len(train_part)} câu | "
        f"dev={len(dev_part)} câu | "
        f"dev chiếm {len(dev_part) / total:.2%} số câu"
    )

print(f"\nTổng train: {len(train_examples)} câu")
print(f"Tổng dev:   {len(dev_examples)} câu")
print(f"Tổng test:  {len(test_examples)} câu")

DrugBank: train=4928 câu | dev=676 câu | dev chiếm 12.06% số câu
MedLine: train=1124 câu | dev=177 câu | dev chiếm 13.60% số câu

Tổng train: 6052 câu
Tổng dev:   853 câu
Tổng test:  665 câu


In [10]:
print(train_examples[67])

{'id': 'DDI-DrugBank.d460.s2', 'document': '/kaggle/input/datasets/tuantc2306/trainingset/DDICorpus/Train/DrugBank/Nateglinide_ddi.xml', 'text': 'Inhibition of CYP3A4 metabolic reactions was not detected in in vitro experiments.', 'entities': []}


In [11]:
def entity_type_counts(examples):
    counter = Counter()
    sent_with_entity = 0
    for ex in examples:
        if ex["entities"]:
            sent_with_entity += 1
        for ent in ex["entities"]:
            counter[ent["type"]] += 1
    return counter, sent_with_entity

train_type_counts, train_sent_with_ent = entity_type_counts(train_examples)
dev_type_counts, dev_sent_with_ent = entity_type_counts(dev_examples)

print("Train entity types:", train_type_counts)
print("Train sentences with entity:", train_sent_with_ent)
print("Test entity types:", dev_type_counts)
print("Test sentences with entity:", dev_sent_with_ent)

Train entity types: Counter({'drug': 8291, 'group': 3067, 'brand': 1252, 'drug_n': 429})
Train sentences with entity: 4900
Test entity types: Counter({'drug': 1141, 'group': 362, 'brand': 185, 'drug_n': 76})
Test sentences with entity: 660


In [12]:
labels = ['O', 'B-brand', 'I-brand', 'B-drug', 'I-drug', 'B-drug_n', 'I-drug_n', 'B-group', 'I-group']
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Train:", len(train_examples), "| Valid:", len(dev_examples), "| Test:", len(test_examples))
print("Labels:", labels)

Train: 6052 | Valid: 853 | Test: 665
Labels: ['O', 'B-brand', 'I-brand', 'B-drug', 'I-drug', 'B-drug_n', 'I-drug_n', 'B-group', 'I-group']


In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(type(tokenizer))
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>
Vocab size: 30522


In [14]:
def align_labels_with_offsets(offsets, entities, label2id):
    token_labels = [label2id["O"] for _ in offsets]
    eval_mask = []

    for idx, (start, end) in enumerate(offsets):
        is_special = (start == 0 and end == 0)
        eval_mask.append(0 if is_special else 1)

    occupied = [False for _ in offsets]
    for ent in sorted(entities, key=lambda x: (x["start"], x["end"])):
        first_token = True
        ent_start, ent_end, ent_type = ent["start"], ent["end"], ent["type"]
        b_label = label2id.get(f"B-{ent_type}", label2id["O"])
        i_label = label2id.get(f"I-{ent_type}", label2id["O"])

        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start == 0 and tok_end == 0:
                continue
            overlaps = tok_start < ent_end and tok_end > ent_start
            if overlaps and not occupied[i]:
                token_labels[i] = b_label if first_token else i_label
                occupied[i] = True
                first_token = False

    return token_labels, eval_mask

In [15]:
class DDINerDataset(Dataset):
    def __init__(self, examples, tokenizer, label2id, max_length=192):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(
            ex["text"],
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            add_special_tokens=True,
        )
        offsets = encoding.pop("offset_mapping")
        labels, eval_mask = align_labels_with_offsets(offsets, ex["entities"], self.label2id)

        return {
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "eval_mask": torch.tensor(eval_mask, dtype=torch.long),
        }

In [16]:
def collate_batch(batch):
    max_len = max(item["input_ids"].size(0) for item in batch)
    pad_id = tokenizer.pad_token_id
    o_id = label2id["O"]

    out = defaultdict(list)
    for item in batch:
        length = item["input_ids"].size(0)
        pad_len = max_len - length
        out["input_ids"].append(torch.cat([item["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        out["attention_mask"].append(torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        out["labels"].append(torch.cat([item["labels"], torch.full((pad_len,), o_id, dtype=torch.long)]))
        out["eval_mask"].append(torch.cat([item["eval_mask"], torch.zeros(pad_len, dtype=torch.long)]))

    return {key: torch.stack(value) for key, value in out.items()}

In [17]:
train_ds = DDINerDataset(train_examples, tokenizer, label2id, MAX_LENGTH)
valid_ds = DDINerDataset(dev_examples, tokenizer, label2id, MAX_LENGTH)
test_ds = DDINerDataset(test_examples, tokenizer, label2id, MAX_LENGTH) 

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=2)
dev_loader = DataLoader(valid_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2) if test_ds else None

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("Decoded sample:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0][:30]))
print("Label sample:", [id2label[i.item()] for i in batch["labels"][0][:30]])

{'input_ids': (32, 58), 'attention_mask': (32, 58), 'labels': (32, 58), 'eval_mask': (32, 58)}
Decoded sample: ['[CLS]', 'no', 'pharmacokinetic', 'data', 'are', 'available', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Label sample: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [18]:
class LinearChainCRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def forward(self, emissions, tags, mask):
        emissions = emissions.float()
        mask = mask.bool()
        log_denominator = self._compute_log_partition(emissions, mask)
        log_numerator = self._compute_score(emissions, tags, mask)
        return torch.mean(log_denominator - log_numerator)

    def _compute_score(self, emissions, tags, mask):
        batch_size, seq_len, _ = emissions.shape
        score = self.start_transitions[tags[:, 0]]
        score += emissions[torch.arange(batch_size, device=emissions.device), 0, tags[:, 0]]

        for t in range(1, seq_len):
            emit_score = emissions[torch.arange(batch_size, device=emissions.device), t, tags[:, t]]
            trans_score = self.transitions[tags[:, t - 1], tags[:, t]]
            score += (emit_score + trans_score) * mask[:, t]

        lengths = mask.long().sum(dim=1).clamp(min=1) - 1
        last_tags = tags[torch.arange(batch_size, device=emissions.device), lengths]
        score += self.end_transitions[last_tags]
        return score

    def _compute_log_partition(self, emissions, mask):
        score = self.start_transitions + emissions[:, 0]
        for t in range(1, emissions.size(1)):
            next_score = score.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, t].unsqueeze(1)
            next_score = torch.logsumexp(next_score, dim=1)
            score = torch.where(mask[:, t].unsqueeze(1), next_score, score)
        score += self.end_transitions
        return torch.logsumexp(score, dim=1)

    @torch.no_grad()
    def decode(self, emissions, mask):
        mask = mask.bool()
        batch_size, seq_len, _ = emissions.shape
        score = self.start_transitions + emissions[:, 0]
        history = []

        for t in range(1, seq_len):
            next_score = score.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, t].unsqueeze(1)
            best_score, best_tag = next_score.max(dim=1)
            score = torch.where(mask[:, t].unsqueeze(1), best_score, score)
            history.append(best_tag)

        score += self.end_transitions
        best_last_tags = score.argmax(dim=1)
        lengths = mask.long().sum(dim=1).clamp(min=1)

        best_paths = []
        for b in range(batch_size):
            seq_len_b = lengths[b].item()
            best_tag = best_last_tags[b].item()
            path = [best_tag]
            for hist in reversed(history[: seq_len_b - 1]):
                best_tag = hist[b, best_tag].item()
                path.append(best_tag)
            path.reverse()
            best_paths.append(path)
        return best_paths


class BertBiLstmCrf(nn.Module):
    def __init__(self, model_name, num_labels, lstm_hidden_size=256, lstm_layers=1, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.0 if lstm_layers == 1 else dropout,
        )
        self.classifier = nn.Linear(lstm_hidden_size * 2, num_labels)
        self.crf = LinearChainCRF(num_labels)

    def emissions(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        lstm_output, _ = self.lstm(sequence_output)
        lstm_output = self.dropout(lstm_output)
        return self.classifier(lstm_output)

    def forward(self, input_ids, attention_mask, labels=None):
        emissions = self.emissions(input_ids, attention_mask)
        if labels is not None:
            loss = self.crf(emissions, labels, attention_mask)
            return loss, emissions
        return emissions

    def decode(self, input_ids, attention_mask):
        emissions = self.emissions(input_ids, attention_mask)
        return self.crf.decode(emissions, attention_mask)

model = BertBiLstmCrf(MODEL_NAME, num_labels=len(labels), lstm_hidden_size=HIDDEN_DIM, dropout=DROPOUT_RATE).to(device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Trainable parameters: 109688236


In [19]:
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import IOB2
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report

def evaluate(model, dataloader, id2label, desc="Evaluating", device="cuda"):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    token_correct = 0
    token_total = 0

    
    all_gold_tags = []
    all_pred_tags = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_tensor = batch["labels"].to(device)
            eval_mask = batch["eval_mask"].to(device)

            loss, _ = model(input_ids, attention_mask, labels_tensor)
            paths = model.decode(input_ids, attention_mask)

            total_loss += loss.item()
            n_batches += 1

            for b, path in enumerate(paths):
                gold_tags = []
                pred_tags = []
                valid_positions = eval_mask[b].bool().cpu().tolist()
                gold_ids = labels_tensor[b].detach().cpu().tolist()

                for pos, keep in enumerate(valid_positions[:len(path)]):
                    if not keep:
                        continue
                    gold = id2label[gold_ids[pos]]
                    pred = id2label[path[pos]]
                    
                    gold_tags.append(gold)
                    pred_tags.append(pred)
                    
                    token_correct += int(gold == pred)
                    token_total += 1

                # Thêm chuỗi nhãn của câu hiện tại vào list tổng
                # Chỉ thêm nếu câu có chứa token hợp lệ (để tránh mảng rỗng)
                if gold_tags:
                    all_gold_tags.append(gold_tags)
                    all_pred_tags.append(pred_tags)

    # ---------------------------------------------------------
    # SỬ DỤNG SEQEVAL ĐỂ TÍNH TOÁN METRICS
    # ---------------------------------------------------------
    
    # 1. Tính Micro F1 (Đồng nhất với cách tính cũ của bạn - gom chung toàn bộ entity)
    micro_precision = precision_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="micro")
    micro_recall = recall_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="micro")
    micro_f1 = f1_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="micro")

    # 2. Tính Macro F1 (Tính F1 riêng cho từng loại entity như PER, ORG, LOC... rồi lấy trung bình cộng)
    macro_f1 = f1_score(all_gold_tags, all_pred_tags, mode="strict", scheme=IOB2, average="macro")

    token_acc = token_correct / token_total if token_total else 0.0
    
    return {
        "loss": float(total_loss / max(n_batches, 1)),
        "token_acc": float(token_acc),
        "precision": float(micro_precision),
        "recall": float(micro_recall),
        "f1": float(micro_f1),
        "macro_f1": float(macro_f1),
    }

In [20]:
import torch
from tqdm import tqdm

def train_epoch(model, train_loader, optimizer, scheduler, scaler, device, epoch, total_epochs, grad_clip):
    model.train()
    train_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{total_epochs} [Train]")
    for step, batch in enumerate(progress, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_tensor = batch["labels"].to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            loss, _ = model(input_ids, attention_mask, labels_tensor)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss += loss.item()
        progress.set_postfix(loss=f"{train_loss / step:.4f}")
    return train_loss / max(len(train_loader), 1)

In [21]:
bert_params = list(model.bert.parameters())
head_params = list(model.lstm.parameters()) + list(model.classifier.parameters()) + list(model.crf.parameters())

optimizer = AdamW(
    [
        {"params": bert_params, "lr": LR_BERT, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
    ]
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
print("Total steps:", total_steps, "| Warmup steps:", warmup_steps)

Total steps: 1900 | Warmup steps: 190


/tmp/ipykernel_58/3610606136.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


In [22]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("training_ddi")

In [23]:
import wandb

WANDB_PROJECT = "ddi-ner-embedAttention_BiLSTM_CRF"
WANDB_RUN_NAME = f"BERT_BiLSTM_CRF, hidden_dim=32"


def wandb_config():
    return {
        "model_name": MODEL_NAME,
        "architecture": "BiomedBERT + BiLSTM + CRF",
        "max_length": MAX_LENGTH,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "epochs": EPOCHS,
        "valid_ratio": VALID_RATIO,
        "lr_bert": LR_BERT,
        "lr_head": LR_HEAD,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "grad_clip": GRAD_CLIP,
        "seed": SEED,
        "num_labels": len(labels),
        "labels": labels,
        "train_sentences": len(train_examples),
        "valid_sentences": len(dev_examples),
        "test_sentences": len(test_examples),
    }


def ensure_wandb_run():
    if wandb.run is None:
        run = wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=wandb_config(),
        )
        wandb.watch(model, log="gradients", log_freq=100)
        return run
    return wandb.run

wandb.login(key=secret_value_0)
wandb_run = ensure_wandb_run()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: noct2306 (noct2306-uet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [24]:
best_dev_macro_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        epoch=epoch,
        total_epochs=EPOCHS,
        grad_clip=GRAD_CLIP
    )
    
    dev_metrics = evaluate(model, dev_loader, id2label, desc=f"Valid eval epoch {epoch}", device=device)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"valid_{k}": v for k, v in dev_metrics.items()},
    }
    history.append(row)
    print(row)

    wandb.log(
        {
            "epoch": epoch,
            "train/loss": train_loss,
            "dev/loss": dev_metrics["loss"],
            "dev/token_acc": dev_metrics["token_acc"],
            "dev/precision": dev_metrics["precision"],
            "dev/recall": dev_metrics["recall"],
            "dev/f1": dev_metrics["f1"],
            "dev/macro_f1": dev_metrics["macro_f1"],
            "lr/bert": optimizer.param_groups[0]["lr"],
            "lr/head": optimizer.param_groups[1]["lr"],
        },
        step=epoch,
    )

    if dev_metrics["macro_f1"] > best_dev_macro_f1:
        best_dev_macro_f1 = dev_metrics["macro_f1"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "model_name": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "valid_metrics": dev_metrics,
            },
            BEST_MODEL_PATH,
        )
        wandb.run.summary["best_dev_macro_f1"] = best_dev_macro_f1
        wandb.run.summary["best_epoch"] = epoch
        wandb.run.summary["best_model_path"] = str(BEST_MODEL_PATH)
        print("Saved best model to", BEST_MODEL_PATH)

history_df = pd.DataFrame(history)
wandb.log({"history": wandb.Table(dataframe=history_df)})
wandb.finish()
history_df

Epoch 1/10 [Train]:   0%|          | 0/190 [00:00<?, ?it/s]/tmp/ipykernel_58/662791565.py:20: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Valid eval epoch 1: 100%|██████████| 27/27 [00:06<00:00,  3.89it/s]


{'epoch': 1, 'train_loss': 23.108664490047254, 'valid_loss': 4.005734746102934, 'valid_token_acc': 0.9535047717549715, 'valid_precision': np.float64(0.794900849858357), 'valid_recall': np.float64(0.7976122797043775), 'valid_f1': np.float64(0.7962542565266741), 'valid_macro_f1': np.float64(0.4269139648950172)}
Saved best model to /kaggle/working/biomedbert_bilstm_crf_ddi_best.pt


Valid eval epoch 2: 100%|██████████| 27/27 [00:07<00:00,  3.61it/s]


{'epoch': 2, 'train_loss': 3.239997012991654, 'valid_loss': 2.1059648195902505, 'valid_token_acc': 0.9763527807813455, 'valid_precision': np.float64(0.9115606936416185), 'valid_recall': np.float64(0.8965321205230244), 'valid_f1': np.float64(0.9039839495557466), 'valid_macro_f1': np.float64(0.7953817733533424)}
Saved best model to /kaggle/working/biomedbert_bilstm_crf_ddi_best.pt


Valid eval epoch 3: 100%|██████████| 27/27 [00:07<00:00,  3.43it/s]


...

{'epoch': 9, 'train_loss': 0.5969056832162958, 'valid_loss': 3.048872373722218, 'valid_token_acc': 0.9757886324103239, 'valid_precision': np.float64(0.9216138328530259), 'valid_recall': np.float64(0.9090392268334281), 'valid_f1': np.float64(0.9152833428734976), 'valid_macro_f1': np.float64(0.8175604365778348)}


Valid eval epoch 10: 100%|██████████| 27/27 [00:07<00:00,  3.50it/s]


{'epoch': 10, 'train_loss': 0.5582067207286232, 'valid_loss': 2.9159251076203807, 'valid_token_acc': 0.9772460157021297, 'valid_precision': np.float64(0.9223245109321059), 'valid_recall': np.float64(0.9113132461625923), 'valid_f1': np.float64(0.9167858164140691), 'valid_macro_f1': np.float64(0.8201601411853503)}


dev/f1,▁▇▇█▇█▇███
dev/loss,█▁▂▁▂▃▅▄▅▄
dev/macro_f1,▁▇▇█▇▇▇▇██
dev/precision,▁▇▇█▇█████
dev/recall,▁▇▇██▇▇▇██
dev/token_acc,▁▇▇█▇▇▇▇▇█
epoch,▁▂▃▃▄▅▆▆▇█
lr/bert,█▇▆▆▅▄▃▃▂▁
lr/head,█▇▆▆▅▄▃▃▂▁
train/loss,█▂▁▁▁▁▁▁▁▁
best_dev_macro_f1,0.84625


,epoch,train_loss,valid_loss,valid_token_acc,valid_precision,valid_recall,valid_f1,valid_macro_f1
0,1,23.108664,4.005735,0.953505,0.794901,0.797612,0.796254,0.426914
1,2,3.239997,2.105965,0.976353,0.911561,0.896532,0.903984,0.795382
2,3,1.923521,2.321145,0.972874,0.903539,0.899943,0.901737,0.771532
3,4,1.433227,2.049540,0.978139,0.920617,0.916430,0.918519,0.846248
4,5,1.102200,2.441090,0.974143,0.905756,0.912450,0.909091,0.814274
5,6,0.925937,2.669765,0.975224,0.920461,0.907902,0.914139,0.814428
6,7,0.767948,3.070496,0.973579,0.913744,0.903354,0.908519,0.799579
7,8,0.689255,2.982183,0.974754,0.917674,0.906197,0.911899,0.813446
8,9,0.596906,3.048872,0.975789,0.921614,0.909039,0.915283,0.817560
9,10,0.558207,2.915925,0.977246,0.922325,0.911313,0.916786,0.820160


In [27]:
# Load best checkpoint before final evaluation.
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"])
print("Best validation metrics:", checkpoint["valid_metrics"])


Best validation metrics: {'loss': 2.0495399214603283, 'token_acc': 0.9781392506229138, 'precision': np.float64(0.9206167904054826), 'recall': np.float64(0.9164297896532121), 'f1': np.float64(0.9185185185185185), 'macro_f1': np.float64(0.8462482629615479)}


In [28]:
test_metrics = evaluate(model, test_loader, id2label, desc=f"Test", device=device)


Test: 100%|██████████| 21/21 [00:05<00:00,  3.95it/s]


In [31]:
print(f"Test loss: {test_metrics['loss']}")
print(f"Test token_acc: {test_metrics['token_acc']}")
print(f"Test precision: {test_metrics['precision']}")
print(f"Test micro_f1: {test_metrics['f1']}")
print(f"Test macro_f1: {test_metrics['macro_f1']}")

Test loss: 2.7170139040265764
Test token_acc: 0.969961518580208
Test precision: 0.7338820301783264
Test micro_f1: 0.7561837455830388
Test macro_f1: 0.7008093049198838


In [37]:
test_drug = Path("/kaggle/input/datasets/tuantc2306/trainingset/DDICorpus/Test/Test for DrugNER task/DrugBank")
test_med = Path("/kaggle/input/datasets/tuantc2306/trainingset/DDICorpus/Test/Test for DrugNER task/MedLine")

test_files_drug = sorted(test_drug.rglob("*.xml"))
test_files_med = sorted(test_med.rglob("*.xml"))

test_examples_drug = load_examples_from_files(test_files_drug)
test_examples_med = load_examples_from_files(test_files_med)

test_ds_drug = DDINerDataset(test_examples_drug, tokenizer, label2id, MAX_LENGTH) 
test_ds_med = DDINerDataset(test_examples_med, tokenizer, label2id, MAX_LENGTH) 

test_loader_drug = DataLoader(test_ds_drug, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2) 
test_loader_med = DataLoader(test_ds_med, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=2) 


In [42]:
test_metrics_drug = evaluate(model, test_loader_drug, id2label, desc=f"Test", device=device)
print(f"Test_Drug loss: {test_metrics_drug['loss']}")
print(f"Test_Drug token_acc: {test_metrics_drug['token_acc']}")
print(f"Test_Drug precision: {test_metrics_drug['precision']}")
print(f"Test_Drug micro_f1: {test_metrics_drug['f1']}")
print(f"Test_Drug macro_f1: {test_metrics_drug['macro_f1']}")

Test: 100%|██████████| 5/5 [00:01<00:00,  3.49it/s]

Test_Drug loss: 2.595511627197266
Test_Drug token_acc: 0.9718232044198895
Test_Drug precision: 0.8896103896103896
Test_Drug micro_f1: 0.8954248366013072
Test_Drug macro_f1: 0.6681990317968731


In [43]:
test_metrics_med = evaluate(model, test_loader_med, id2label, desc=f"Test", device=device)
print(f"Test_Med loss: {test_metrics_med['loss']}")
print(f"Test_Med token_acc: {test_metrics_med['token_acc']}")
print(f"Test_Med precision: {test_metrics_med['precision']}")
print(f"Test_Med micro_f1: {test_metrics_med['f1']}")
print(f"Test_Med macro_f1: {test_metrics_med['macro_f1']}")

Test: 100%|██████████| 17/17 [00:04<00:00,  4.08it/s]


Test_Med loss: 2.925623532603769
Test_Med token_acc: 0.9694728446088029
Test_Med precision: 0.6199524940617577
Test_Med micro_f1: 0.6500622665006227
Test_Med macro_f1: 0.619743732471544
